In [2]:
from miditok import REMI
from pathlib import Path
TOKENIZER_PATH = Path("../configs/tokenizer.json")

tokenizer = REMI(params=TOKENIZER_PATH)

c:\Users\aqdas\Minor_Project\piano-music-composition\.venv\Lib\site-packages\miditok\classes.py:886: UserWarning: Argument nb_tempos has been renamed num_tempos, you should consider to updateyour code with this new argument name.
  return cls(**input_dict, **kwargs)


In [3]:

import torch
from transformers import AutoModelForCausalLM


MODEL_NAME = "Natooz/Maestro-REMI-bpe20k"
device = "cuda" if torch.cuda.is_available() else "cpu"

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
).to(device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:

baseline_model.eval()

input_ids = torch.tensor(
    [[baseline_model.config.bos_token_id]],
    dtype=torch.long,
    device=device,
)

with torch.no_grad():
    baseline_ids = baseline_model.generate(
        input_ids,
        max_new_tokens=2000,
        do_sample=True,
        temperature=2.5,
        top_k=15,
        top_p=0.95,
        pad_token_id=baseline_model.config.eos_token_id,
    )

baseline_ids = baseline_ids[0].tolist()
baseline_midi = tokenizer.decode([baseline_ids])

output_path = Path("../outputs/pretrained/unprompted.mid")
output_path.parent.mkdir(parents=True, exist_ok=True)
baseline_midi.dump_midi(str(output_path))

print("Saved baseline to:", output_path.resolve())


Saved baseline to: C:\Users\aqdas\Minor_Project\piano-music-composition\outputs\pretrained\unprompted.mid
